In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
%pip install ultralytics>=8.3.185
import ultralytics
ultralytics.checks()

Ultralytics 8.3.237  Python-3.10.19 torch-2.9.1+cpu CPU (AMD Ryzen 5 7600X 6-Core Processor)
Setup complete  (12 CPUs, 31.2 GB RAM, 2582.7/3566.4 GB disk)


In [14]:
from ultralytics import YOLO

In [13]:
import sys
print(sys.executable)

c:\Users\rkobj\anaconda3\python.exe


In [14]:
!conda create -n rf python=3.10 -y
!conda activate rf
!pip install --upgrade pip
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="3W8f4p5waQiNXYHFEDuW")
project = rf.workspace("juwan-kim").project("6class-wehlm-ptoky-vot4x")
version = project.version(1)
dataset = version.download("yolov8")
                

Channels:
 - defaults
Platform: win-64
Solving environment: ...working... done

## Package Plan ##

  environment location: C:\Users\rkobj\anaconda3\envs\rf

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  bzip2              pkgs/main/win-64::bzip2-1.0.8-h2bbff1b_6 
  ca-certificates    pkgs/main/win-64::ca-certificates-2025.12.2-haa95532_0 
  expat              pkgs/main/win-64::expat-2.7.3-h885b0b7_4 
  libexpat           pkgs/main/win-64::libexpat-2.7.3-h885b0b7_4 
  libffi             pkgs/main/win-64::libffi-3.4.4-hd77b12b_1 
  libzlib            pkgs/main/win-64::libzlib-1.3.1-h02ab6af_0 
  openssl            pkgs/main/win-64::openssl-3.0.18-h543e019_0 
  pip                pkgs/main/noarch::pip-25.3-pyhc872135_0 
  python             pkgs/main/win-64::python-3.10.19-h981015d_0 
  setuptools         pkgs/main/win-64::setuptools-80.9.0-py310haa95532_0 
  sqlite             pkgs/main/win-64::sqlite-3.51.0-hda9a48d_0 
  tk               

ERROR: To modify pip, please run the following command:
C:\Users\rkobj\anaconda3\python.exe -m pip install --upgrade pip


  Using cached roboflow-1.2.11-py3-none-any.whl.metadata (9.7 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached roboflow-1.2.11-py3-none-any.whl (89 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.10.0


error: uninstall-no-record-file

× Cannot uninstall opencv-python-headless 4.10.0
╰─> The package's contents are unknown: no RECORD file was found for opencv-python-headless.

hint: The package was installed by conda. You should check if it can uninstall the package.


ModuleNotFoundError: No module named 'roboflow'

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
import yaml

# -----------------------------
# 🧩 경로 설정
# -----------------------------
base_dir = "/kaggle/input/new-new-data/train"  # 원본 YOLO 데이터셋 경로
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

output_dir = "/kaggle/working/split_yolo"  # 결과 저장 경로

# -----------------------------
# 🗂 출력 폴더 구조 생성
# -----------------------------
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

# -----------------------------
# 📸 전체 이미지 파일 목록
# -----------------------------
images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
images.sort()

# -----------------------------
# 📊 데이터 분할 (70:15:15)
# -----------------------------
train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

def copy_split(img_list, split_name):
    for img in img_list:
        label = os.path.splitext(img)[0] + ".txt"
        src_img = os.path.join(images_dir, img)
        src_lbl = os.path.join(labels_dir, label)
        dst_img = os.path.join(output_dir, "images", split_name, img)
        dst_lbl = os.path.join(output_dir, "labels", split_name, label)

        shutil.copy(src_img, dst_img)
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)

copy_split(train_imgs, "train")
copy_split(val_imgs, "val")
copy_split(test_imgs, "test")

print(f"✅ 데이터 분할 완료! ({len(train_imgs)} train / {len(val_imgs)} val / {len(test_imgs)} test)")

# -----------------------------
# 🧾 data.yaml 자동 생성
# -----------------------------
yaml_path = os.path.join(output_dir, "data.yaml")

data_config = {
    "train": os.path.join(output_dir, "images/train"),
    "val": os.path.join(output_dir, "images/val"),
    "test": os.path.join(output_dir, "images/test"),
    "nc": 6,  # 클래스 개수 수정 필요
    "names": ['CCS1-AC', 'CCS1-DC', 'CCS2-AC', 'CCS2-DC', 'GB-AC', 'GB-DC']
}

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"✅ data.yaml 생성 완료: {yaml_path}")


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: '/kaggle/input/new-new-data/train\\images'

In [ ]:

model = YOLO("yolov8s.pt")

model.train(
    data="/kaggle/working/split_yolo/data.yaml",
    imgsz=640,
    epochs=50
    
)


In [ ]:
metrics = model.val(data="/kaggle/working/split_yolo/data.yaml")  # no arguments needed, dataset and settings remembered
metrics.box.map

metrics_test = model.val(data="/kaggle/working/split_yolo/data.yaml",split='test')
metrics_test.box.map